# KNN imputer: filling Titanic `Age` from similar passengers

Run each cell in order with **Shift + Enter**. Each step matches a section of the Note.
The data (`data/titanic_knn.csv`) is six columns of the Kaggle Titanic `train.csv` (891 passengers): `Age`, `Pclass`, `Fare`, `SibSp`, `Parch` and the target `Survived`. Sections 2 to 6 use `Age`, `Pclass` and `Fare`; section 7 adds `SibSp` and `Parch`.

## 1. The worked example: nan-Euclidean distance by hand and by scikit-learn

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import nan_euclidean_distances

# the 5-row table of the Note; row 2 (index 1) has a gap in f1
toy = np.array([[30, 60, np.nan],
                [np.nan, 55, 20],
                [40, 52, 25],
                [25, np.nan, 22],
                [50, 70, 40]])

# distance from row 2 to every row (0 to itself)
nan_euclidean_distances(toy[[1]], toy).round(2)

array([[ 8.66,  0.  ,  7.14,  3.46, 30.62]])

In [2]:
# the same distance by hand, for row 2 and row 3
a, b = toy[1], toy[2]
both = ~np.isnan(a) & ~np.isnan(b)                # columns where both rows have a value
weight = len(a) / both.sum()                      # 3 columns / 2 used = 1.5
np.sqrt(weight * ((a[both] - b[both]) ** 2).sum())   # 7.14

np.float64(7.14142842854285)

In [3]:
from sklearn.impute import KNNImputer

# fill the gap from the 2 nearest rows: plain mean, then distance-weighted
print(KNNImputer(n_neighbors=2).fit_transform(toy)[1])
print(KNNImputer(n_neighbors=2, weights="distance").fit_transform(toy)[1])

[32.5 55.  20. ]
[29.89947452 55.         20.        ]


## 2. Load the Titanic data and split it

In [4]:
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/titanic_knn.csv")
df.isnull().mean() * 100          # share of missing values per column: only Age has gaps

Age         19.86532
Pclass       0.00000
Fare         0.00000
SibSp        0.00000
Parch        0.00000
Survived     0.00000
dtype: float64

In [5]:
X = df[["Age", "Pclass", "Fare"]]   # inputs
y = df["Survived"]                   # target

# split first: the imputer may only learn from the training rows
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
X_train.head()

,Age,Pclass,Fare
30,40.0,1,27.7208
10,4.0,3,16.7000
873,47.0,3,9.0000
182,9.0,3,31.3875
876,20.0,3,9.8458


## 3. KNN imputation, then logistic regression

In [6]:
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# lbfgs warns that it needs more iterations on the unscaled Fare column; the result is the same
warnings.filterwarnings("ignore")

knn = KNNImputer(n_neighbors=3, weights="distance")
X_train_trf = knn.fit_transform(X_train)   # fit: remember the training rows; transform: fill their gaps
X_test_trf = knn.transform(X_test)         # test gaps are filled from training neighbours only

lr = LogisticRegression()
lr.fit(X_train_trf, y_train)
accuracy_score(y_test, lr.predict(X_test_trf))

0.7039106145251397

## 4. Compare with mean imputation

In [7]:
from sklearn.impute import SimpleImputer

si = SimpleImputer()                       # strategy="mean"
X_train_trf2 = si.fit_transform(X_train)
X_test_trf2 = si.transform(X_test)

lr = LogisticRegression()
lr.fit(X_train_trf2, y_train)
accuracy_score(y_test, lr.predict(X_test_trf2))

0.6927374301675978

## 5. Try every k and both weightings

In [8]:
def accuracy(imputer):
    # fit the imputer and the model on the training set, score on the test set
    model = LogisticRegression().fit(imputer.fit_transform(X_train), y_train)
    return accuracy_score(y_test, model.predict(imputer.transform(X_test)))

results = pd.DataFrame({w: [accuracy(KNNImputer(n_neighbors=k, weights=w)) for k in range(1, 11)]
                        for w in ["uniform", "distance"]}, index=range(1, 11))
results.index.name = "k"
results.round(3)

,uniform,distance
k,,
1,0.721,0.721
2,0.709,0.709
3,0.704,0.704
4,0.704,0.704
5,0.704,0.704
6,0.704,0.704
7,0.704,0.704
8,0.704,0.704
9,0.704,0.704


In [9]:
# one test passenger is worth 1/179 of accuracy
print(len(X_test), round(1 / len(X_test), 4))

179 0.0056


## 6. Adding a missing indicator

In [10]:
knn_ind = KNNImputer(n_neighbors=3, weights="distance", add_indicator=True)
out = knn_ind.fit_transform(X_train)
print(knn_ind.get_feature_names_out())   # the extra column marks the rows whose Age was missing
print(accuracy(knn_ind))

['Age' 'Pclass' 'Fare' 'missingindicator_Age']
0.7039106145251397


In [11]:
# Test: is Fare choosing the neighbours? Drop Pclass, or scale first, and count how many filled ages change
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
gaps = X_train["Age"].isna().values
a = KNNImputer(n_neighbors=3, weights="distance").fit_transform(X_train)[gaps, 0]
b = KNNImputer(n_neighbors=3, weights="distance").fit_transform(X_train.drop(columns="Pclass"))[gaps, 0]
sc = StandardScaler().fit(X_train)   # skips NaN when it fits
c = sc.inverse_transform(KNNImputer(n_neighbors=3, weights="distance").fit_transform(sc.transform(X_train)))[gaps, 0]
print("same fill without Pclass:", np.isclose(a, b).mean().round(2))
print("same fill after scaling:", np.isclose(a, c).mean().round(2))
p = make_pipeline(StandardScaler(), KNNImputer(n_neighbors=3, weights="distance"), LogisticRegression()).fit(X_train, y_train)
print("test accuracy with scaling:", round(accuracy_score(y_test, p.predict(X_test)), 3))

same fill without Pclass: 0.81
same fill after scaling: 0.26


test accuracy with scaling: 0.704


## 7. Is the KNN fill closer to the truth? Hide known ages and compare

We take the 714 passengers whose `Age` is known, hide the ages of a random 20%, fill them, and measure the mean absolute error (in years) against the true ages. Features: `Pclass`, `Fare`, `SibSp`, `Parch`, scaled first. Averaged over 20 random splits.

In [12]:
from sklearn.metrics import mean_absolute_error

known = df.loc[df["Age"].notna(), ["Age", "Pclass", "Fare", "SibSp", "Parch"]].reset_index(drop=True)

def fill_error(k=None, weights="uniform", scale=True, seeds=range(20)):
    # k=None means mean imputation; returns the mean absolute error of the filled ages, averaged over seeds
    errors = []
    for s in seeds:
        tr, te = train_test_split(known, test_size=0.2, random_state=s)
        hidden = te.copy()
        hidden["Age"] = np.nan                                   # hide the true ages
        if k is None:
            filled = np.full(len(te), tr["Age"].mean())
        else:
            sc = StandardScaler().fit(tr) if scale else StandardScaler(with_mean=False, with_std=False).fit(tr)
            imp = KNNImputer(n_neighbors=k, weights=weights).fit(sc.transform(tr))
            filled = sc.inverse_transform(imp.transform(sc.transform(hidden)))[:, 0]
        errors.append(mean_absolute_error(te["Age"], filled))
    return np.mean(errors)

print("mean imputation:", round(fill_error(None), 2), "years")
print("KNN, k=10:      ", round(fill_error(10), 2), "years")
print("KNN, k=10, no scaling:", round(fill_error(10, scale=False), 2), "years")

mean imputation: 11.24 years


KNN, k=10:       9.76 years


KNN, k=10, no scaling: 10.66 years


In [13]:
# fill error for each k and both weightings (Figure 4 of the Note)
ks = [1, 2, 3, 5, 7, 10, 15, 20]
errs = pd.DataFrame({w: [fill_error(k, w) for k in ks] for w in ["uniform", "distance"]}, index=ks)
errs.index.name = "k"
errs.round(2)

,uniform,distance
k,,
1,12.52,12.52
2,10.84,11.47
3,10.48,11.11
5,10.10,10.97
7,9.94,10.86
10,9.76,10.79
15,9.80,10.73
20,9.78,10.71


In [14]:
# does a better fill change the model? Logistic regression accuracy, averaged over 20 splits
X5 = df[["Age", "Pclass", "Fare", "SibSp", "Parch"]]
def mean_accuracy(imputer, cols=X5.columns):
    acc = []
    for s in range(20):
        a, b, ya, yb = train_test_split(X5[cols], y, test_size=0.2, random_state=s)
        steps = [StandardScaler()] + ([imputer] if imputer is not None else []) + [LogisticRegression()]
        acc.append(make_pipeline(*steps).fit(a, ya).score(b, yb))
    return round(np.mean(acc), 3)

print("mean imputation:", mean_accuracy(SimpleImputer()))
print("KNN, k=10:      ", mean_accuracy(KNNImputer(n_neighbors=10)))
print("no Age at all:  ", mean_accuracy(None, ["Pclass", "Fare", "SibSp", "Parch"]))

mean imputation: 0.706


KNN, k=10:       0.703


no Age at all:   0.689
